# RootNav2 pilot — ABRS & APEX-03 root angles vs the 18-way skew assay

**Goal.** Quantify root **growth angles** in the spaceflight root images and compare them to the
nutrient-driven skew angles already measured in `data/rsml/18_way_skew/` (RootNav v1).
The skew assays were run to help interpret these flight-vs-ground root phenotypes.

**This is a PILOT** — run on 2-3 frames first and decide whether automatic segmentation is good
enough before scaling up.

### Read these caveats first
1. **Hard target.** Dense (~15-20+ seedlings/plate), overlapping/crossing roots. RootNav2's automatic
   seed->primary->lateral tracing will mis-assign where roots cross, worst on flight (microgravity waving).
2. **Image-quality ranking:** APEX-03 ground-control (color, white roots, high contrast, printed grid)
   > APEX-03 flight > ABRS ground (monochrome green) > ABRS flight. **Start with APEX-03 GC.**
3. **Method confound.** Skew data was traced with RootNav **v1** (semi-automatic). Using RootNav **2.0**
   here mixes a tool difference into comparisons. Prefer **scale/method-robust traits**: angular
   dispersion, tip/emergence angle, tortuosity. Validate a few traces by eye.
4. **Calibration.** Skew RSML is uncalibrated (`unit=pixel`). APEX-03's grid gives px->mm if needed;
   **angles need no calibration.**

### Two paths
- **Path A (RootNav2):** clone + run RootNav 2.0 -> RSML, then parse angles. Needs PyTorch + model.
- **Path B (fallback, runnable now):** classic CV — threshold -> skeletonize -> local orientation ->
  **root-angle distribution**. Robust to overlap (never separates individual roots).

The comparison half (parsing the 18-way skew RSML) runs immediately with no model.


## 0. Setup


In [ ]:
# !pip install numpy pandas matplotlib scikit-image
import os, glob, re, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pathlib import Path
from skimage.io import imread

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand/'data'/'rsml'/'18_way_skew').exists() or (cand/'18_way_skew').exists():
            return cand
    return p
REPO = find_repo_root(); print('repo root:', REPO)

SKEW_DIR = REPO/'data'/'rsml'/'18_way_skew'
if not SKEW_DIR.exists(): SKEW_DIR = REPO/'18_way_skew'

def first_existing(*cands):
    for c in cands:
        if (REPO/c).exists(): return REPO/c
    return REPO/cands[0]
APEX_DIR = first_existing('data/images/apex03','APEX03')
ABRS_GC  = first_existing('data/images/abrs_timelapse/ABRS_Ground_11_days_11_photos',
                          'ABRS_NASA_Roots_TimeLapse/ABRS_Ground_11_days_11_photos')
RESULTS = REPO/'results'/'rootnav2_pilot'; RESULTS.mkdir(parents=True, exist_ok=True)
print('skew RSML :', SKEW_DIR, SKEW_DIR.exists())
print('APEX-03   :', APEX_DIR, APEX_DIR.exists())
print('ABRS GC   :', ABRS_GC, ABRS_GC.exists())


## 1. Pick 2-3 pilot frames
APEX-03 ground-control first (cleanest); an ABRS GC frame included for contrast.


In [ ]:
apex = sorted(glob.glob(str(APEX_DIR/'*GC*.JPG'))) + sorted(glob.glob(str(APEX_DIR/'*GC*.jpg')))
abrs = sorted(glob.glob(str(ABRS_GC/'*.jpg')))
pilot = (apex[:2] + abrs[:1]) if apex else abrs[:3]
print('pilot frames:')
for p in pilot: print('  ', os.path.relpath(p, REPO))

fig, axes = plt.subplots(1, len(pilot), figsize=(5*len(pilot),5)); axes=np.atleast_1d(axes)
for ax, p in zip(axes, pilot):
    ax.imshow(imread(p)); ax.set_title(os.path.basename(p)[:28], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Path B (runnable now) — root-angle distribution via classic CV
Segment root pixels -> skeletonize -> local orientation -> **angle distribution**. Its dispersion is a
method-robust proxy for skew/wave (ground roots cluster near vertical; flight roots spread out).


In [ ]:
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu, sobel_h, sobel_v, gaussian
from skimage.morphology import skeletonize, remove_small_objects

def root_angles(img_path, invert=None, smooth=2.0, min_size=200):
    im = imread(img_path)
    g = rgb2gray(im) if im.ndim==3 else im/255.0
    g = gaussian(g, smooth)
    t = threshold_otsu(g)
    fg_light, fg_dark = (g > t), (g < t)
    mask = fg_light if fg_light.mean() < fg_dark.mean() else fg_dark  # roots = thinner -> fewer px
    if invert is True:  mask = ~mask
    if invert is False: mask = fg_light
    mask = remove_small_objects(mask, min_size)
    skel = skeletonize(mask)
    gx, gy = sobel_v(g), sobel_h(g)
    ang = (np.degrees(np.arctan2(gx, gy)) + 90) % 180     # structure orientation 0..180
    a = ang[skel]; a = np.where(a > 90, a - 180, a)        # fold to -90..90 (0 = vertical)
    return a, mask, skel

rows=[]
fig, axes = plt.subplots(2, len(pilot), figsize=(5*len(pilot),9)); axes=np.atleast_2d(axes)
for j,p in enumerate(pilot):
    a, mask, skel = root_angles(p)
    rows.append(dict(frame=os.path.basename(p), n_px=int(len(a)),
                     mean_abs_angle=float(np.mean(np.abs(a))) if len(a) else float('nan'),
                     angle_std=float(np.std(a)) if len(a) else float('nan')))
    axes[0,j].imshow(skel, cmap='gray'); axes[0,j].set_title('skeleton '+os.path.basename(p)[:16], fontsize=8); axes[0,j].axis('off')
    axes[1,j].hist(a, bins=37, range=(-90,90), color='seagreen'); axes[1,j].set_xlabel('angle from vertical (deg)')
plt.tight_layout(); plt.show()
cv_df = pd.DataFrame(rows); cv_df.to_csv(RESULTS/'pathB_cv_angle_summary.csv', index=False)
print(cv_df.to_string(index=False)); print('wrote', RESULTS/'pathB_cv_angle_summary.csv')
# Tune invert/min_size/smooth per image set — this is a starting point, not a tuned pipeline.


## 3. Path A (scaffold) — RootNav 2.0 -> RSML
https://github.com/robail-yasrab/RootNav-2.0 (Yasrab et al. 2019). Needs PyTorch + pretrained model;
**expect to fine-tune** for these dense plates.


In [ ]:
# --- SCAFFOLD: uncomment & adapt; run on a GPU (e.g. Colab GPU runtime). ---
# !git clone https://github.com/robail-yasrab/RootNav-2.0.git
# %cd RootNav-2.0 && pip install -r requirements.txt
# # See repo README for the exact CLI; it writes one .rsml per input image:
# # !python inference/predict.py --model <pretrained> --input <pilot_dir> --output <out_dir>
ROOTNAV2_OUT = RESULTS/'rootnav2_rsml'   # set to RootNav2's output dir after it runs
print('Set ROOTNAV2_OUT, then run sections 4 & 6. exists:', ROOTNAV2_OUT.exists())


## 4. Parse RSML -> angle traits (skew baseline AND RootNav2 output use the SAME reader)


In [ ]:
def _poly_points(root_el):
    pts = root_el.findall('.//polyline/point') or root_el.findall('.//point')
    return [(float(p.get('x')), float(p.get('y'))) for p in pts]

def angle_from_vertical(p0, p1):
    dx, dy = p1[0]-p0[0], p1[1]-p0[1]   # image y grows downward; seed(top)->tip(bottom)
    return math.degrees(math.atan2(dx, dy))   # 0 = straight down

def root_traits(pts):
    if len(pts) < 2: return None
    seg = max(3, len(pts)//10)
    emergence = angle_from_vertical(pts[0], pts[min(seg,len(pts)-1)])
    tip       = angle_from_vertical(pts[-min(seg,len(pts))], pts[-1])
    overall   = angle_from_vertical(pts[0], pts[-1])
    path = sum(math.dist(pts[i],pts[i+1]) for i in range(len(pts)-1))
    chord = math.dist(pts[0], pts[-1]) or float('nan')
    return dict(emergence_angle=emergence, tip_angle=tip, primary_angle=overall,
                path_len_px=path, tortuosity=path/chord)

def rsml_to_df(rsml_path):
    root = ET.parse(rsml_path).getroot(); out=[]
    for ri, r in enumerate(root.findall('.//root')):
        t = root_traits(_poly_points(r))
        if t: out.append(dict(file=os.path.basename(rsml_path), root_idx=ri, **t))
    return pd.DataFrame(out)

demo = sorted(glob.glob(str(SKEW_DIR/'*.rsml')))[0]
print(rsml_to_df(demo).head().to_string(index=False))


## 5. Build the 18-way skew angle baseline


In [ ]:
def parse_treatment(fn):
    m=re.match(r'Gradient_([\d.]+)%([AP])_([\d.]+)%S_(\d+)', fn)
    if not m: return {}
    g,agent,s,rep=m.groups()
    return dict(gelling_agent={'A':'agar','P':'phytogel'}[agent], gelling_pct=float(g),
                sucrose_pct=float(s), replicate=int(rep))

skew=[]
for f in sorted(glob.glob(str(SKEW_DIR/'*.rsml'))):
    d=rsml_to_df(f)
    for k,v in parse_treatment(os.path.basename(f)).items(): d[k]=v
    skew.append(d)
skew=pd.concat(skew, ignore_index=True)
skew.to_csv(RESULTS/'skew_root_traits.csv', index=False)
print('skew roots:', len(skew))
print(skew.groupby('gelling_agent')[['primary_angle','tip_angle','tortuosity']].mean().to_string())

fig,ax=plt.subplots(1,2,figsize=(12,4))
for agent,grp in skew.groupby('gelling_agent'):
    ax[0].hist(grp['primary_angle'].dropna(), bins=40, range=(-90,90), alpha=0.5, label=agent)
ax[0].set_title('Skew primary-root angle by gelling agent'); ax[0].set_xlabel('angle from vertical (deg)'); ax[0].legend()
skew.boxplot(column='tortuosity', by='sucrose_pct', ax=ax[1]); ax[1].set_title('Tortuosity by sucrose %'); plt.suptitle('')
plt.tight_layout(); plt.show()


## 6. Compare flight vs ground vs skew
Load RootNav2 RSML (sections 3-4) and overlay angle distributions on the skew baseline.


In [ ]:
frames_traits=[]
if 'ROOTNAV2_OUT' in globals() and Path(ROOTNAV2_OUT).exists():
    for f in glob.glob(str(Path(ROOTNAV2_OUT)/'*.rsml')):
        d=rsml_to_df(f); d['source']='rootnav2'; frames_traits.append(d)
if frames_traits:
    flt=pd.concat(frames_traits, ignore_index=True)
    flt.to_csv(RESULTS/'flight_ground_root_traits.csv', index=False)
    plt.hist(skew['primary_angle'].dropna(), bins=40, range=(-90,90), density=True, alpha=0.4, label='skew baseline')
    plt.hist(flt['primary_angle'].dropna(),  bins=40, range=(-90,90), density=True, alpha=0.4, label='flight/ground')
    plt.legend(); plt.xlabel('primary-root angle from vertical (deg)'); plt.title('Angle distributions'); plt.show()
else:
    print('No RootNav2 RSML yet. Run section 3, set ROOTNAV2_OUT, re-run.')
    print('Meanwhile Path B (section 2) gives a model-free angle-dispersion comparison.')


## 7. Notes & next steps
- **Decision gate:** if Path A on APEX-03 GC gives clean traces (spot-check by eye), scale up. If not,
  use RootNav **v1 semi-automatic** (matches the skew method) or rely on **Path B** distributions.
- **Calibrate** APEX-03 via its printed grid only if absolute lengths are needed (angles don't need it).
- **Stats:** compare angle dispersion / mean|angle| / tortuosity across flight vs ground vs the 18 skew
  treatments (`skew_root_traits.csv`).
- **Provenance:** export new traces as RSML; add a datapackage entry mirroring
  `data/rsml/18_way_skew/datapackage.json`.
- See `PROJECT_TRACKER.md` Tier 7.
